# benchmarks.md — чужие модели, бенчмарки и бюджеты (внешний контекст)

## 1. Два разных зачёта на Atari

Одну и ту же игру проверяют в двух несовместимых режимах — это ключ ко всем сравнениям.

| | **Atari-57** («марафон») | **Atari-100k** («спринт») |
|---|---|---|
| Вопрос | «насколько силён, если учиться не жалко» | «насколько быстро учится на крохах» |
| Игр | 57 | 26 (Frostbite входит) |
| Бюджет | 200 млн кадров на игру (920 ч игры) | 100k шагов = 400k кадров (~2 ч игры) |
| Что мерят | потолок мастерства | сэмпл-эффективность |
| Заведён | — (классический протокол ALE) | SimPLe, Kaiser et al. 2019 |

Зачёт Atari-100k ввели, чтобы уйти от гонки «у кого больше железа»: 100k шагов среды — это
два часа реальной игры (каждые 4 кадра эмулятора сливаются в один шаг, `action repeat=4`).

## 2. Метрика: Human Normalized Score (HNS)

```
HNS = (очки_агента − очки_случайной_игры) / (очки_человека − очки_случайной_игры)
```

- **0** — уровень случайной игры (тыканье кнопок);
- **1.0** — уровень «человека» (по протоколу ALE — средний игрок после ~2 ч знакомства);
- **> 1.0** — «сверхчеловек» по этой мерке.

⚠️ Средний HNS по 26/57 играм прячет конкретику: на отдельных играх картина может быть
обратной (см. Frostbite ниже — там почти все SOTA *ниже* человека по HNS).

Различать: **mean HNS**, **median HNS**, **IQM** (interquartile mean), **capped HNS**
(обрезанный сверху) — разные агрегаты дают разные рейтинги.

## 3. Шкала Frostbite (игра «жёсткого исследования»)

| Ориентир | Очки | Источник |
|---|---|---|
| Случайная игра | 65.2 | arXiv:2112.04145 (A Review for Deep RL in Atari) |
| «Человек» (бенчмарк ALE, ~2 ч) | 4334.7 | там же (каноничный «Machado human») |

Порядок «~2–2.5к очков на уровень» согласуется с нашими наблюдениями, но линейной шкалы
«уровень ↔ очки» у нас нет — это прикидка.

## 4. Карточки: «спринтеры» Atari-100k

### 4.1. EfficientZero V2 (2024) — лидер 100k, **2.428** mean HNS (median 1.286)

- **Тип:** модельный, с поиском (MCTS в обученной модели среды).
- **Архитектура:** representation/dynamics/policy/value-сети + **Sampling-based Gumbel Search**
  (замена MCTS, работает и в непрерывном действии); temporal-consistency loss.
- **Параметры:** меньше, чем у BBF; ниже replay ratio.
- **Бюджет:** Atari-100k, вход 96×96, 4 кадра стека.
- **Источник:** arXiv:2403.00564 (ICML 2024).

### 4.2. BBF — Bigger, Better, Faster (2023) — **2.247** mean HNS

- **Тип:** безмодельный, value-based.
- **Архитектура:** Impala-CNN / ResNet для оценки value; масштабирование ширины слоёв.
- **Параметры:** вилка 0.15 → 21.8 млн (ResNet), 0.20 → 32.9 млн (CNN); ширина ×4 к базовой Impala.
- **Бюджет:** 100k шагов, ~**6 ч на одной GPU**.
- **Источник:** arXiv:2305.19452 (ICML 2023).

### 4.3. DIAMOND (2024) — **1.46** mean HNS

- **Тип:** модельный, обучение целиком «в воображении» (imagination-only).
- **Архитектура:** **диффузионная world model на U-Net**; двухступенчатый пайплайн
  (динамика в низком разрешении → апсемплинг).
- **Параметры:** **~4.4 млн** для Atari; масштабировано до **381 млн** (вкл. 51 млн апсемплер) на CS:GO.
- **Источник:** arXiv:2405.12399 (NeurIPS 2024).

### 4.4. TWISTER (ICLR 2025) — **162%** (1.62) mean HNS

- **Тип:** модельный, трансформерная world model. **Без look-ahead search** (рекорд среди таких).
- **Архитектура:** три сети — world model (трансформер), actor, critic; ключ —
  **action-conditioned Contrastive Predictive Coding** (InfoNCE-стиль) вместо простого next-state.
- **Источник:** arXiv:2503.04416; код github.com/burchim/TWISTER.

## 5. Карточки: «марафонцы» Atari-57

### 5.1. Agent57 (2020) — первый выше человека на всех 57 играх

- **Тип:** безмодельный, распределённый.
- **Архитектура:** R2D2 (recurrent, LSTM) + NGU (episodic memory на kNN + intrinsic reward) +
  UVFA (семья политик: разный exploration/exploitation, 32 политики).
- **Бюджет:** **~78 млрд кадров** до обхода человека на всех 57.
- **Источник:** arXiv:2003.13350 (ICML 2020).

### 5.2. MEME — Human-level Atari 200x faster (2022)

- **Тип:** безмодельный, прямая переработка Agent57.
- **Архитектура:** та же линия (R2D2/NGU/UVFA) + нормализация value разных масштабов,
  trust-region для value, NFNets-приёмы, policy distillation.
- **Бюджет:** **390 млн кадров** — тот же результат (все 57 выше человека), ~200× экономнее.
- **Источник:** arXiv:2209.07550. (Таблица очков по играм — Appendix M, Table 5.)

### 5.3. Для сравнения (mean HNS, 200M, из обзора)

MuZero ≈4996%; Muesli ≈2539%; R2D2 ≈3374%; NGU ≈3170%; и т.д. — см. arXiv:2112.04145.
Ключ: MuZero даёт высокие uncapped mean/median, но низкий capped (провалы на части игр).

## 6. Карточки: ветка «модель мира»

### 6.1. DreamerV3 (Nature 2025) — универсальность

- **Тип:** модельный, обучение в воображении.
- **Архитектура:** **RSSM** — детерминированный `h` + стохастические категориальные латенты `z`;
  symlog-наблюдения, KL-баланс с free bits, 1% unimix, actor-critic в латенте.
- **Параметры:** вилка 12 → 400 млн.
- **Особенность:** **фиксированные гиперпараметры** без подгонки под задачу; >150 задач.
- **Источник:** Nature 640:647–653, 2025 (arXiv:2301.04104).

### 6.2. Dreamer 4 (2025)

- **Тип:** модельный, обучение целиком в модели мира; Minecraft.
- **Архитектура:** causal video tokenizer (masked autoencoder) + **эффективный трансформер**;
  новый objective **shortcut forcing** (стабильность на длинном горизонте).
- **Параметры:** ~**2 млрд** (400M токенайзер + 1.6B динамика).
- **Источник:** arXiv:2509.24527.

## 7. Карточка: RL как моделирование последовательности

### 7.1. Decision Transformer (2021)

- **Тип:** offline RL как последовательностное моделирование (без value-функций и policy gradient).
- **Архитектура:** **GPT** (causal self-attention); вход — тройки (return-to-go, state, action)
  через модальные эмбеддинги + positional/timestep encoding.
- **Параметры (Atari, на базе minGPT):** 6 слоёв, 8 голов, эмбеддинг 128; контекст K=30–50; batch 512.
- **Бюджет:** offline, на датасетах; без обучения в среде.
- **Источник:** arXiv:2106.01345 (NeurIPS 2021).

## 8. Сырые очки Frostbite (для справки)

**Atari-100k** (из таблицы EfficientZero V2, arXiv:2403.00564):

| Метод | Очки Frostbite | HNS ≈ |
|---|---|---|
| Случайно | 65.2 | 0.00 |
| MuZero | 255.0 | 0.04 |
| SimPLe | 254.7 | 0.04 |
| EfficientZero | 296.3 | 0.05 |
| DrQ | 331.1 | 0.06 |
| DreamerV3 | 909 | 0.20 |
| EZ-V2 | 1136.3 | 0.25 |
| SPR | 1821.5 | 0.41 |
| BBF | 2384.8 | 0.54 |
| Человек | 4334.7 | 1.00 |

HNS пересчитан по формуле (random 65.2, human 4334.7).

**Atari-57** (бюджет в кадрах среды, из Human-level Atari 200x faster):

| Метод | Кадров среды | Frostbite, очки |
|---|---|---|
| Agent57 @ 1B | 1 млрд | 980.15 |
| Agent57 @ 90B | 90 млрд | 507 775.65 |
| MEME @ 200M | 200 млн | 136 691.77 |
| MEME @ 1B | 1 млрд | 526 239.50 |

Вывод: на Frostbite «марафонцы» перепрыгивают человека только при сотнях миллионов —
миллиардах кадров. Agent57 на 1 млрд кадров даёт лишь 980 очков (человек — 4334).

## 9. Источники (ссылки)

1. A Review for Deep RL in Atari: Benchmarks, Challenges, Solutions — arXiv:2112.04145
2. EfficientZero V2 — arXiv:2403.00564
3. BBF: Bigger, Better, Faster — arXiv:2305.19452
4. DIAMOND: Diffusion for World Modeling — arXiv:2405.12399
5. TWISTER: Learning Transformer-based World Models with CPC — arXiv:2503.04416
6. Agent57: Outperforming the Atari Human Benchmark — arXiv:2003.13350
7. Human-level Atari 200x faster (MEME) — arXiv:2209.07550
8. DreamerV3: Mastering diverse control tasks through world models — Nature 640:647–653, 2025 / arXiv:2301.04104
9. Dreamer 4: Training Agents Inside of Scalable World Models — arXiv:2509.24527
10. Decision Transformer — arXiv:2106.01345